# Palimpsest — ChangeFormer Fine-tune (A100)

**Ejecutar Cell 1 primero en cada sesión.**

| | |
|---|---|
| **Cell 1** | Setup / restore desde GCS |
| **Cell 2** | TTA sweep — test gratis, sin entrenar |
| **Cell 3** | Fine-tune 4: patch 512px, 50 épocas (~25 min) |
| **Cell 4** | Evaluación final con TTA |

---
**Historial**

| Run | Epochs | pw | LR | Patch | Test F1 |
|---|---|---|---|---|---|
| Initial | 1–100 | 10 | 6e-5 | 256 | — |
| FT-1 | 101–175 | 4 | 2e-5 | 256 | — |
| FT-2 | 176–250 | 3 | 1e-5 | 256 | 0.8120 |
| FT-3 | 251–325 | 2.5 | 5e-6 | 256 | **0.8169** |
| FT-4 | 326–375 | 3 | 3e-5 | **512** | — |

## 1 · Setup / restore (run this first every session)

Installs packages, authenticates, downloads code + dataset, restores checkpoint from GCS.

In [ ]:
# ── 1a. Packages ─────────────────────────────────────────────────────────────
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm', 'einops', 'pydantic-settings', 'scipy', 'Pillow',
                'google-cloud-storage', 'google-cloud-bigquery',
                'rasterio', 'earthengine-api'], check=True)
print('Packages ready.')

# ── 1b. Auth ─────────────────────────────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
print('Authenticated.')

WORKDIR = '/content/palimpsest'
CKPT    = f'{WORKDIR}/checkpoints/changeformer_levir.pth'

def _run(cmd):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), check=True,
                       capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.strip())

# ── 1c. Code ─────────────────────────────────────────────────────────────────
if not os.path.isdir(f'{WORKDIR}/model'):
    print('Downloading code …')
    os.makedirs(WORKDIR, exist_ok=True)
    _run(['gsutil', 'cp',
          'gs://palimpsest-bucket/code/palimpsest_code.tar.gz', WORKDIR])
    _run(f'tar xzf {WORKDIR}/palimpsest_code.tar.gz -C {WORKDIR} '
         f'&& rm {WORKDIR}/palimpsest_code.tar.gz')
    with open(f'{WORKDIR}/.env', 'w') as f:
        f.write('GCP_PROJECT_ID=project-8c7ca821-aa7a-45ea-88b\n'
                'GCS_BUCKET_NAME=palimpsest-bucket\n'
                'EE_PROJECT=project-8c7ca821-aa7a-45ea-88b\n')
    print('  Code ready.')
else:
    print('Code present.')

# ── 1d. Dataset ───────────────────────────────────────────────────────────────
val_a = f'{WORKDIR}/data/datasets/val/A'
if not os.path.isdir(val_a) or len(os.listdir(val_a)) < 60:
    print('Downloading dataset (all splits) …')
    os.makedirs(f'{WORKDIR}/data/datasets', exist_ok=True)
    subprocess.run(['gsutil', '-m', 'cp', '-r',
                    'gs://palimpsest-bucket/datasets/*',
                    f'{WORKDIR}/data/datasets/'], check=True)
    from pathlib import Path
    for sp in ('train', 'val', 'test'):
        n = len(list(Path(f'{WORKDIR}/data/datasets/{sp}/A').glob('*.png')))
        print(f'  {sp}: {n} pairs')
else:
    print(f'Dataset present ({len(os.listdir(val_a))} val pairs).')

# ── 1e. Checkpoint ────────────────────────────────────────────────────────────
os.makedirs(f'{WORKDIR}/checkpoints', exist_ok=True)
if not os.path.exists(CKPT):
    print('Restoring checkpoint from GCS …')
    _run(['gsutil', 'cp',
          'gs://palimpsest-bucket/checkpoints/changeformer_levir.pth', CKPT])
    print(f'  Restored ({os.path.getsize(CKPT)/1e6:.1f} MB).')
else:
    print(f'Checkpoint present ({os.path.getsize(CKPT)/1e6:.1f} MB).')

# ── 1f. Python path ───────────────────────────────────────────────────────────
os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

# Verify GPU
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
else:
    print('WARNING: no GPU — Runtime → Change runtime type → T4 GPU')

print(f'\nReady.  cwd={os.getcwd()}')

## 2 · TTA sweep — ganancia gratis

**Test-Time Augmentation**: promedia predicciones sobre 4 orientaciones (original + hflip + vflip + ambas).  
No requiere entrenamiento. Esperado: +1–2% F1 sobre los 0.8169 actuales.

In [ ]:
# ── 2a. Sweep en val SIN TTA (baseline) ──────────────────────────────────────
print("=== Val sin TTA ===")
proc = subprocess.Popen(
    [sys.executable, 'model/evaluate.py',
     '--checkpoint', CKPT, '--data-root', f'{WORKDIR}/data/datasets',
     '--split', 'val', '--threshold-sweep'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=WORKDIR)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

# ── 2b. Sweep en val CON TTA ─────────────────────────────────────────────────
print("\n=== Val CON TTA ===")
proc = subprocess.Popen(
    [sys.executable, 'model/evaluate.py',
     '--checkpoint', CKPT, '--data-root', f'{WORKDIR}/data/datasets',
     '--split', 'val', '--threshold-sweep', '--tta'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=WORKDIR)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

# ── 2c. Test set CON TTA al mejor threshold ──────────────────────────────────
BEST_THR_TTA = 0.60   # ← actualiza con el resultado de 2b
print(f"\n=== Test CON TTA (threshold={BEST_THR_TTA}) ===")
proc = subprocess.Popen(
    [sys.executable, 'model/evaluate.py',
     '--checkpoint', CKPT, '--data-root', f'{WORKDIR}/data/datasets',
     '--split', 'test', '--fast', '--tta', '--threshold', str(BEST_THR_TTA)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=WORKDIR)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

## 3 · Fine-tune 4 — patch 512px, 50 épocas (~25 min)

La F1 se ha estabilizado en 0.82 con patch=256. El motivo es **falta de contexto**: 256px = ~2.5 km².  
Con 512px el modelo ve 4× más área por crop y distingue mejor el fondo de cambios reales.

Con A100:
- Batch = 8 a 512×512 (comfortable en 40 GB)
- ~30 batches/epoch → ~25 s/epoch → 50 épocas = **~21 min**
- LR sube a 3e-5 para "despertar" el modelo ante la nueva escala, con warmup corto

> **Nota:** el checkpoint guarda las métricas del val con patch=256.  
> El best_f1 del checkpoint (0.8172) puede ser superado ya en la época 5–10 al evaluar con patch=512.

In [ ]:
cmd = [
    sys.executable, 'model/train.py',
    '--epochs',       '375',      # 326-375 = 50 épocas nuevas
    '--batch-size',   '8',        # 512×512 en A100 40GB
    '--lr',           '3e-5',     # LR más alto para adaptarse al nuevo tamaño
    '--warmup',       '3',
    '--pos-weight',   '3',
    '--workers',      '4',
    '--val-interval', '5',
    '--patch-size',   '512',      # ← contexto 4× mayor
    '--data-root', f'{WORKDIR}/data/datasets',
    '--checkpoint', CKPT,
    '--resume',
    '--gcs-upload',
]
print('Fine-tune 4: epochs 326–375  patch=512  batch=8  lr=3e-5  pos_weight=3\n')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, cwd=WORKDIR)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExited {proc.returncode}')

## 4 · Evaluación final — test set con TTA y patch 512

Actualiza `BEST_THR` con el mejor threshold del sweep de cell 2.

In [ ]:
BEST_THR = 0.60   # ← actualiza desde cell 2

# Sweep post-FT4 en val con patch=512 y TTA
print("=== Val post-FT4 (patch=512, TTA) ===")
proc = subprocess.Popen(
    [sys.executable, 'model/evaluate.py',
     '--checkpoint', CKPT, '--data-root', f'{WORKDIR}/data/datasets',
     '--split', 'val', '--threshold-sweep', '--tta', '--patch-size', '512'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=WORKDIR)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

# Test final
print(f"\n=== Test FINAL (patch=512, TTA, threshold={BEST_THR}) ===")
proc = subprocess.Popen(
    [sys.executable, 'model/evaluate.py',
     '--checkpoint', CKPT, '--data-root', f'{WORKDIR}/data/datasets',
     '--split', 'test', '--fast', '--tta',
     '--patch-size', '512', '--threshold', str(BEST_THR)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=WORKDIR)
for line in proc.stdout: print(line, end='', flush=True)
proc.wait()

## 5 · Final test-set evaluation

Set `BEST_THRESHOLD` from the sweep above.

In [ ]:
BEST_THRESHOLD = 0.5   # ← update from sweep (cell 2 / 4)

proc = subprocess.Popen(
    [sys.executable, 'model/evaluate.py',
     '--checkpoint', CKPT,
     '--data-root',  f'{WORKDIR}/data/datasets',
     '--split', 'test',
     '--fast',
     '--threshold', str(BEST_THRESHOLD)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=WORKDIR,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

## 6 · Download checkpoint (optional)
Already at `gs://palimpsest-bucket/checkpoints/`.

In [ ]:
from google.colab import files
files.download(CKPT)